In [4]:
import py3Dmol

In [26]:
import py3Dmol
v = py3Dmol.view()
v.addModel(open(f"6o4w_REDO.pdb").read())
v.setStyle({'chain':'A'}, {'cartoon': {'color': '#0e9674'}})
v.setStyle({'chain':'B'}, {'cartoon': {'color': '#c46225'}})
v.zoomTo({'model':0})
v.rotate(90, "z")
v.rotate(-25, "y")

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [2]:
from pdbfixer import PDBFixer
from openmm.app import PDBFile

In [3]:
# Load PDB file directly
fixer = PDBFixer(filename='6o4w_REDO.pdb')
fixer.removeChains(chainIds=['A']) #Removing chain A, retaining chain B
fixer.findMissingResidues()
fixer.findNonstandardResidues()
fixer.replaceNonstandardResidues()
fixer.removeHeterogens(keepWater=False)
fixer.findMissingAtoms()
fixer.addMissingAtoms()
fixer.addMissingHydrogens(7.4)

In [34]:
PDBFile.writeFile(fixer.topology, fixer.positions, open('prepped_protein/6O4W_REDO_Fixed_ChainB.pdb', 'w'))


In [4]:
import py3Dmol
v = py3Dmol.view()
v.addModel(open(f"prepped_protein/6O4W_REDO_Fixed_ChainB.pdb").read())
v.setStyle({'chain':'A'}, {'cartoon': {'color': '#0e9674'}})
v.setStyle({'chain':'B'}, {'cartoon': {'color': '#c46225'}})
v.zoomTo({'model':0})
v.rotate(90, "z")
v.rotate(-25, "y")

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [40]:
os.chdir("/root/Buccheri_trial/Buccheri raw files")

In [9]:
ligand_resname = 'E20'
ligand_chain = 'B'

with open('6o4w_REDO.pdb', 'r') as f_in, open('selected_ligands/6O4W_ligand_E20_B.pdb', 'w') as f_out:
    for line in f_in:
        if (line.startswith('HETATM') and
            line[17:20].strip() == ligand_resname and
            line[21].strip() == ligand_chain):
            f_out.write(line)

print(f"Ligand {ligand_resname} from chain {ligand_chain} saved as 6O4W_REDO_Fixed_ChainB.pdb")

Ligand E20 from chain B saved as 6O4W_REDO_Fixed_ChainB.pdb


In [10]:
v = py3Dmol.view() ##view smiles chain
v.addModel(open('selected_ligands/6O4W_ligand_E20_B.pdb').read())
v.setStyle({'stick':{'colorscheme':'greenCarbon'}})
v.zoomTo()
v.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [11]:
!~/gnina -r prepped_protein/6O4W_REDO_Fixed_ChainB.pdb -l selected_ligands/6O4W_ligand_E20_B.pdb --autobox_ligand selected_ligands/6O4W_ligand_E20_B.pdb --seed 0 --exhaustiveness 16 -o redocked_E20.sdf

              _             
             (_)            
   __ _ _ __  _ _ __   __ _ 
  / _` | '_ \| | '_ \ / _` |
 | (_| | | | | | | | | (_| |
  \__, |_| |_|_|_| |_|\__,_|
   __/ |                    
  |___/                     

gnina v1.3.2 master:f23dd2b   Built Jul 29 2025.
gnina is based on smina and AutoDock Vina.
Please cite appropriately.

Commandline: /root/gnina -r prepped_protein/6O4W_REDO_Fixed_ChainB.pdb -l selected_ligands/6O4W_ligand_E20_B.pdb --autobox_ligand selected_ligands/6O4W_ligand_E20_B.pdb --seed 0 --exhaustiveness 16 -o redocked_E20.sdf
*** Open Babel Warning  in Init
  Cannot initialize database 'space-groups.txt' which may cause further errors.
Using random seed: 0

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

mode |  affinity  |  intramol  |    CNN     |   CNN
     | (kcal/mol) | (kcal/mol) | pose score | affinity
-----+------------+---------

In [49]:
pdb_chain_pairs = [
    ("2XYN.pdb", ["C"]),
    ("4ht2_final.pdb", ["A"]),
    ("3emg_final.pdb", ["A"]),
    ("4djw_final.pdb", ["B"]),
    ("1KE9.pdb", ["A"]),
    ("5olh_final.pdb", ["A"]),
    ("7BVQ.pdb", ["B"]),
    ("4o09_final.pdb", ["A"]),
    ("5edu_final.pdb", ["B"])
]

for pdb_file, chains_to_keep in pdb_chain_pairs:
    # Load PDB
    fixer = PDBFixer(filename=pdb_file)
    
    # Remove chains not in chains_to_keep
    all_chains = [c.id for c in fixer.topology.chains()]
    chains_to_remove = [c for c in all_chains if c not in chains_to_keep]
    fixer.removeChains(chainIds=chains_to_remove)
    
    # Standard cleaning
    fixer.findMissingResidues()
    fixer.findNonstandardResidues()
    fixer.replaceNonstandardResidues()
    fixer.removeHeterogens(keepWater=False)
    fixer.findMissingAtoms()
    fixer.addMissingAtoms()
    fixer.addMissingHydrogens(pH=7.4)
    
    # Save cleaned PDB to prepped_protein folder
    chains_str = "_".join(chains_to_keep)
    output_file = os.path.join("prepped_protein", f"{pdb_file.split('.')[0]}_Fixed_Chain{chains_str}.pdb")
    with open(output_file, 'w') as f_out:
        PDBFile.writeFile(fixer.topology, fixer.positions, f_out)
    
    print(f"Processed {pdb_file} → saved chain(s) {chains_str} as {output_file}")


Processed 2XYN.pdb → saved chain(s) C as prepped_protein/2XYN_Fixed_ChainC.pdb
Processed 4ht2_final.pdb → saved chain(s) A as prepped_protein/4ht2_final_Fixed_ChainA.pdb
Processed 3emg_final.pdb → saved chain(s) A as prepped_protein/3emg_final_Fixed_ChainA.pdb
Processed 4djw_final.pdb → saved chain(s) B as prepped_protein/4djw_final_Fixed_ChainB.pdb
Processed 1KE9.pdb → saved chain(s) A as prepped_protein/1KE9_Fixed_ChainA.pdb
Processed 5olh_final.pdb → saved chain(s) A as prepped_protein/5olh_final_Fixed_ChainA.pdb
Processed 7BVQ.pdb → saved chain(s) B as prepped_protein/7BVQ_Fixed_ChainB.pdb
Processed 4o09_final.pdb → saved chain(s) A as prepped_protein/4o09_final_Fixed_ChainA.pdb
Processed 5edu_final.pdb → saved chain(s) B as prepped_protein/5edu_final_Fixed_ChainB.pdb


In [7]:
import os

# Map of PDB files to the chain(s) and ligands you want to extract
# Only 2XYN specifies a residue number
pdb_ligand_map = {
    "2XYN.pdb": {
        "chains": ["C"],
        "ligands": [{"resname": "VX6", "resnum": "547"}]
    },
    "4ht2_final.pdb": {
        "chains": ["A"],
        "ligands": [{"resname": "V50", "resnum": None}]
    },
    "3emg_final.pdb": {
        "chains": ["A"],
        "ligands": [{"resname": "685", "resnum": None}]
    },
    "4djw_final.pdb": {
        "chains": ["B"],
        "ligands": [{"resname": "0KP", "resnum": None}]
    },
    "1KE9.pdb": {
        "chains": ["A"],
        "ligands": [{"resname": "LS5", "resnum": None}]
    },
    "5olh_final.pdb": {
        "chains": ["A"],
        "ligands": [{"resname": "9XT", "resnum": None}]
    },
    "7BVQ.pdb": {
        "chains": ["B"],
        "ligands": [{"resname": "CAU", "resnum": None}]
    },
    "4o09_final.pdb": {
        "chains": ["A"],
        "ligands": [{"resname": "2R6", "resnum": None}]
    },
    "5edu_final.pdb": {
        "chains": ["B"],
        "ligands": [{"resname": "TSN", "resnum": None}]
    }
}

output_dir = "selected_ligands"
os.makedirs(output_dir, exist_ok=True)

for pdb_file, info in pdb_ligand_map.items():
    chains_to_keep = info["chains"]

    for ligand in info["ligands"]:
        resname = ligand["resname"]
        resnum = ligand["resnum"]

        output_file = os.path.join(
            output_dir,
            f"{pdb_file.split('.')[0]}_ligand_{resname}_{'_'.join(chains_to_keep)}.pdb"
        )

        with open(pdb_file, "r") as f_in, open(output_file, "w") as f_out:
            for line in f_in:
                if not line.startswith("HETATM"):
                    continue

                if line[17:20].strip() != resname:
                    continue

                if line[21].strip() not in chains_to_keep:
                    continue

                # Only applied for 2XYN (VX6 residue 547)
                if resnum is not None and line[22:26].strip() != resnum:
                    continue

                f_out.write(line)

        print(
            f"Saved ligand {resname}"
            + (f" residue {resnum}" if resnum else "")
            + f" from {pdb_file} chain(s) {chains_to_keep}"
        )


Saved ligand VX6 residue 547 from 2XYN.pdb chain(s) ['C']
Saved ligand V50 from 4ht2_final.pdb chain(s) ['A']
Saved ligand 685 from 3emg_final.pdb chain(s) ['A']
Saved ligand 0KP from 4djw_final.pdb chain(s) ['B']
Saved ligand LS5 from 1KE9.pdb chain(s) ['A']
Saved ligand 9XT from 5olh_final.pdb chain(s) ['A']
Saved ligand CAU from 7BVQ.pdb chain(s) ['B']
Saved ligand 2R6 from 4o09_final.pdb chain(s) ['A']
Saved ligand TSN from 5edu_final.pdb chain(s) ['B']


In [16]:
import subprocess
import pandas as pd
import os

# =========================
# CONFIG
# =========================
GNINA = "~/gnina"  # remove the !, subprocess handles execution
MODE = 1  # use pose 1 consistently

# Folders
protein_dir = "prepped_protein"
ligand_dir = "selected_ligands"
ideal_ligand_dir = "ideal_ligands"  # NEW folder for ideal ligands

pairs = [
    {"pdb": "6O4W_REDO_Fixed_ChainB.pdb", "ligand": "6O4W_ligand_E20_B.pdb", "ideal": "E20_Ideal.sdf", "lig_id": "E20"},
    {"pdb": "1KE9_Fixed_ChainA.pdb", "ligand": "1KE9_ligand_LS5_A.pdb", "ideal": "LS5_Ideal.sdf", "lig_id": "LS5"},
    {"pdb": "2XYN_Fixed_ChainC.pdb", "ligand": "2XYN_ligand_VX6_C.pdb", "ideal": "VX6_Ideal.sdf", "lig_id": "VX6"},
    {"pdb": "3emg_final_Fixed_ChainA.pdb", "ligand": "3emg_final_ligand_685_A.pdb", "ideal": "685_Ideal.sdf", "lig_id": "685"},
    {"pdb": "4djw_final_Fixed_ChainB.pdb", "ligand": "4djw_final_ligand_0KP_B.pdb", "ideal": "0KP_Ideal.sdf", "lig_id": "0KP"},
    {"pdb": "4ht2_final_Fixed_ChainA.pdb", "ligand": "4ht2_final_ligand_V50_A.pdb", "ideal": "V50_Ideal.sdf", "lig_id": "V50"},
    {"pdb": "4o09_final_Fixed_ChainA.pdb", "ligand": "4o09_final_ligand_2R6_A.pdb", "ideal": "2R6_Ideal.sdf", "lig_id": "2R6"},
    {"pdb": "5edu_final_Fixed_ChainB.pdb", "ligand": "5edu_final_ligand_TSN_B.pdb", "ideal": "TSN_Ideal.sdf", "lig_id": "TSN"},
    {"pdb": "5olh_final_Fixed_ChainA.pdb", "ligand": "5olh_final_ligand_9XT_A.pdb", "ideal": "9XT_Ideal.sdf", "lig_id": "9XT"},
    {"pdb": "7BVQ_Fixed_ChainB.pdb", "ligand": "7BVQ_ligand_CAU_B.pdb", "ideal": "CAU_Ideal.sdf", "lig_id": "CAU"},
]

# =========================
# HELPERS
# =========================
def run_cmd(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print("COMMAND FAILED:\n", cmd)
        print(result.stderr)
        raise RuntimeError("Execution stopped")
    return result.stdout


def parse_cnn_from_table(output, mode=1):
    for line in output.splitlines():
        line = line.strip()
        if not line:
            continue
        if line.startswith(str(mode)):
            parts = [p for p in line.split() if p.replace('.', '', 1).replace('-', '', 1).isdigit()]
            if len(parts) < 3:
                continue
            vina_affinity = float(parts[1])
            cnn_pose = float(parts[3])
            cnn_affinity = float(parts[4])
            cnn_vs = cnn_pose * cnn_affinity
            return vina_affinity, cnn_pose, cnn_affinity, cnn_vs
    return None, None, None, None


def get_rmsd(ref, sdf, mode=1):
    out = run_cmd(f"obrms -f {ref} {sdf}")
    rmsds = [float(l.split()[-1]) for l in out.splitlines() if l.strip().startswith("RMSD")]
    return rmsds[mode - 1] if len(rmsds) >= mode else None

# =========================
# DOCKING LOOP
# =========================
results = []

for p in pairs:
    lig = p["lig_id"]
    print(f"\n🚀 Docking {lig} into {p['pdb']}")

    # Full paths for protein and ligand
    receptor_file = os.path.join(protein_dir, p['pdb'])
    ligand_file   = os.path.join(ligand_dir, p['ligand'])
    ideal_file    = os.path.join(ideal_ligand_dir, p['ideal'])  # NEW folder

    # ---- REDOCK ----
    redock_out = f"redocked_{lig}.sdf"
    out = run_cmd(
        f"{GNINA} -r {receptor_file} "
        f"-l {ligand_file} "
        f"--autobox_ligand {ligand_file} "
        f"--seed 0 --exhaustiveness 16 "
        f"-o {redock_out}"
    )

    vina, pose, aff, vs = parse_cnn_from_table(out, MODE)
    rmsd = get_rmsd(ligand_file, redock_out, MODE)

    results.append({
        "protein": p["pdb"],
        "ligand": lig,
        "dock_type": "redock",
        "vina_affinity": vina,
        "CNNpose": pose,
        "CNNaffinity": aff,
        "CNN_VS": vs,
        "RMSD": rmsd
    })

    # ---- IDEAL ----
    ideal_out = f"docked_{lig}_ideal.sdf"
    out = run_cmd(
        f"{GNINA} -r {receptor_file} "
        f"-l {ideal_file} "
        f"--autobox_ligand {ligand_file} "
        f"--seed 0 --exhaustiveness 16 "
        f"-o {ideal_out}"
    )

    vina, pose, aff, vs = parse_cnn_from_table(out, MODE)
    rmsd = get_rmsd(ligand_file, ideal_out, MODE)

    results.append({
        "protein": p["pdb"],
        "ligand": lig,
        "dock_type": "ideal",
        "vina_affinity": vina,
        "CNNpose": pose,
        "CNNaffinity": aff,
        "CNN_VS": vs,
        "RMSD": rmsd
    })

# =========================
# RESULTS
# =========================
df = pd.DataFrame(results)
df



🚀 Docking E20 into 6O4W_REDO_Fixed_ChainB.pdb

🚀 Docking LS5 into 1KE9_Fixed_ChainA.pdb

🚀 Docking VX6 into 2XYN_Fixed_ChainC.pdb

🚀 Docking 685 into 3emg_final_Fixed_ChainA.pdb

🚀 Docking 0KP into 4djw_final_Fixed_ChainB.pdb

🚀 Docking V50 into 4ht2_final_Fixed_ChainA.pdb

🚀 Docking 2R6 into 4o09_final_Fixed_ChainA.pdb

🚀 Docking TSN into 5edu_final_Fixed_ChainB.pdb

🚀 Docking 9XT into 5olh_final_Fixed_ChainA.pdb

🚀 Docking CAU into 7BVQ_Fixed_ChainB.pdb


,protein,ligand,dock_type,vina_affinity,CNNpose,CNNaffinity,CNN_VS,RMSD
0,6O4W_REDO_Fixed_ChainB.pdb,E20,redock,-12.27,0.9705,7.685,7.458292,0.507244
1,6O4W_REDO_Fixed_ChainB.pdb,E20,ideal,-12.09,0.9666,7.689,7.432187,0.650776
2,1KE9_Fixed_ChainA.pdb,LS5,redock,-10.15,0.9746,6.769,6.597067,1.880300
3,1KE9_Fixed_ChainA.pdb,LS5,ideal,-9.52,0.9385,6.509,6.108697,1.848360
4,2XYN_Fixed_ChainC.pdb,VX6,redock,-9.16,0.9558,8.494,8.118565,1.029590
5,2XYN_Fixed_ChainC.pdb,VX6,ideal,-9.97,0.9941,8.640,8.589024,0.901028
6,3emg_final_Fixed_ChainA.pdb,685,redock,-8.98,0.9853,7.935,7.818355,1.065830
7,3emg_final_Fixed_ChainA.pdb,685,ideal,-9.01,0.9853,7.963,7.845944,1.443710
8,4djw_final_Fixed_ChainB.pdb,0KP,redock,-8.99,0.9715,6.807,6.613001,0.945091
9,4djw_final_Fixed_ChainB.pdb,0KP,ideal,-7.74,0.8936,6.537,5.841463,1.846890


In [41]:
from datetime import datetime
from pathlib import Path

# -----------------------------
# Metadata
# -----------------------------
from datetime import datetime
from pathlib import Path

# -----------------------------
# Metadata
# -----------------------------
now = datetime.now()
report_time = now.strftime("%Y-%m-%d %H:%M:%S")

# Unique experiment ID generated each run (Unix timestamp)
experiment_id = int(now.timestamp())

# -----------------------------
# Output directory
# -----------------------------
results_dir = Path("results")
results_dir.mkdir(exist_ok=True)

outfile = results_dir / f"buccheri_validation_{experiment_id}.txt"

# -----------------------------
# Buccheri reference table
# -----------------------------
buccheri_refs = {
    "6O4W": ("Buccheri_6O4W", 0.92, 7.29, 1.71),
    "1KE9": ("Buccheri_1KE9", 0.97, 6.52, 1.80),
    "2XYN": ("Buccheri_2XYN", 0.99, 8.47, 0.79),
    "3emg": ("Buccheri_3emg", 0.99, 7.85, 0.97),
    "4djw": ("Buccheri_4djw", 0.98, 6.83, 0.44),
    "4ht2": ("Buccheri_4ht2", 0.90, 7.73, 1.37),
    "4o09": ("Buccheri_4o09", 0.96, 7.75, 1.05),
    "5edu": ("Buccheri_5edu", 0.97, 6.88, 1.80),
    "5olh": ("Buccheri_5olh", 0.96, 7.60, 0.29),
    "7BVQ": ("Buccheri_7BVQ", 0.98, 7.56, 1.69),
}

# -----------------------------
# Static header + metadata
# -----------------------------
header = f"""Report produced {report_time} experiment id: {experiment_id}

Run Buccheri et al. (2025) protein-ligand pairs to validate workflow using both redocked ligands and ideal structures.
Updated to include a pH of 7.4 for prepared proteins

File preparation descriptions:
name            functionname  description
PDBfixfunc      TBD           PDBFixer package used to isolate chains of interest and fix the protein with following options:
                               findMissingResidues(), findNonstandardResidues(), replaceNonstandardResidues(),
                               fixer.removeHeterogens(keepWater=False), findMissingAtoms(), addMissingAtoms(),
                               addMissingHydrogens(7.4)

Proteins:
id     name                              source      prep
6O4W   Acetylcholinesterase              PDB-Redo    PDBfixfunc
1KE9   Cyclin-dependent kinase 2         RCSB PDB    PDBfixfunc
2XYN   Tyrosine-protein kinase ABL2      RCSB PDB    PDBfixfunc
3emg   SYK kinase                        PDB-Redo    PDBfixfunc
4djw   Beta-secretase 1                  PDB-Redo    PDBfixfunc
4ht2   Carbonic anhydrase II             PDB-Redo    PDBfixfunc
4o09   HSP90 alpha                       PDB-Redo    PDBfixfunc
5edu   HDAC6                             PDB-Redo    PDBfixfunc
5olh   Adenosine A2a receptor            PDB-Redo    PDBfixfunc
7BVQ   Dopamine D3 receptor              RCSB PDB    PDBfixfunc


Ligands:
id           source                                   prep                     name
E20 (redock) extracted from 6o4w_final_redo_chainB.pdb                        1-BENZYL-4-[(5,6-DIMETHOXY-1-INDANON-2-YL)METHYL]PIPERIDINE
E20 (ideal)  PubChem                                                          1-BENZYL-4-[(5,6-DIMETHOXY-1-INDANON-2-YL)METHYL]PIPERIDINE
LS5 (redock) extracted from 1KE9_Fixed_ChainA.pdb                             3-{{[4-([AMINO(IMINO)METHYL]AMINOSULFONYL)ANILINO]METHYLENE}}-2-OXO-2,3-DIHYDRO-1H-INDOLE
LS5 (ideal)  PubChem                                                          3-{{[4-([AMINO(IMINO)METHYL]AMINOSULFONYL)ANILINO]METHYLENE}}-2-OXO-2,3-DIHYDRO-1H-INDOLE
VX6 (redock) extracted from 2XYN_Fixed_ChainC.pdb (residue 547)               CYCLOPROPANECARBOXYLIC ACID {{4-[4-(4-METHYL-PIPERAZIN-1-YL)-6-(5-METHYL-2H-PYRAZOL-3-YLAMINO)-PYRIMIDIN-2-YLSULFANYL]-PHENYL}}-AMIDE
VX6 (ideal)  PubChem                                                          CYCLOPROPANECARBOXYLIC ACID {{4-[4-(4-METHYL-PIPERAZIN-1-YL)-6-(5-METHYL-2H-PYRAZOL-3-YLAMINO)-PYRIMIDIN-2-YLSULFANYL]-PHENYL}}-AMIDE
TSN (ideal)  PubChem                                                          TRICHOSTATIN A


Gnina options:
exhaustiveness 16


Results:
proteinid + ligandid @ ligandid_searchspace         CNN Score     CNN_VS       RMSD
"""

# -----------------------------
# Build results rows
# -----------------------------
rows = []
added_refs = set()

for _, r in df.iterrows():
    prot_full = r["protein"]
    prot_key = prot_full.split("_")[0]

    # Buccheri reference row
    if prot_key in buccheri_refs and prot_key not in added_refs:
        ref_label, ref_cnn, ref_vs, ref_rmsd = buccheri_refs[prot_key]
        rows.append(
            f"{ref_label:<55} {ref_cnn:>10.4f} {ref_vs:>12.6f} {ref_rmsd:>12.6f}"
        )
        added_refs.add(prot_key)

    label = f"{prot_full} + {r['ligand']} ({r['dock_type']})"
    cnn_score = r["CNNpose"]
    cnn_vs = r["CNN_VS"]
    rmsd = "inf" if r["RMSD"] == float("inf") else f"{r['RMSD']:.6f}"

    rows.append(
        f"{label:<55} {cnn_score:>10.4f} {cnn_vs:>12.6f} {rmsd:>12}"
    )

# -----------------------------
# Write report
# -----------------------------
report = header + "\n".join(rows)

with open(outfile, "w", encoding="utf-8") as f:
    f.write(report)

print(report)
print(f"\nSaved report to: {outfile.resolve()}")


Report produced 2026-02-06 16:01:19 experiment id: 1770422479

Run Buccheri et al. (2025) protein-ligand pairs to validate workflow using both redocked ligands and ideal structures.
Updated to include a pH of 7.4 for prepared proteins

File preparation descriptions:
name            functionname  description
PDBfixfunc      TBD           PDBFixer package used to isolate chains of interest and fix the protein with following options:
                               findMissingResidues(), findNonstandardResidues(), replaceNonstandardResidues(),
                               fixer.removeHeterogens(keepWater=False), findMissingAtoms(), addMissingAtoms(),
                               addMissingHydrogens(7.4)

Proteins:
id     name                              source      prep
6O4W   Acetylcholinesterase              PDB-Redo    PDBfixfunc
1KE9   Cyclin-dependent kinase 2         RCSB PDB    PDBfixfunc
2XYN   Tyrosine-protein kinase ABL2      RCSB PDB    PDBfixfunc
3emg   SYK kinase         